# CaptionBERT rotary mechanism diagnostics
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AbstractEyes/captionbert-trainer/blob/v0.2.0/notebooks/captionbert_rotary_diagnostics.ipynb)

Use a **separate CPU runtime** for this notebook. The existing GPU training notebook
continues unchanged. Install tagged v0.2.0 in its own environment, then measure only
completed frozen checkpoints. No training or consensus-bank download occurs.
Each checkpoint is at most 300 MB; the fixed probe set has 16 texts of at most 256
tokens. Two CPU threads are used. Enable HF_TOKEN to upload the additive JSON reports.


In [ ]:
def run_logged(command, log_path, env=None):
    import os
    from pathlib import Path
    import subprocess
    import time

    child_env = os.environ.copy() if env is None else dict(env)
    child_env["PYTHONUNBUFFERED"] = "1"
    child_env["PYTHONIOENCODING"] = "utf-8"
    path = Path(log_path)
    path.parent.mkdir(parents=True, exist_ok=True)
    print(f"Streaming child output; console log: {path}", flush=True)
    started = time.monotonic()
    with path.open("a", encoding="utf-8", buffering=1) as log:
        process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                   text=True, encoding="utf-8", errors="replace", bufsize=1,
                                   env=child_env)
        try:
            for line in process.stdout:
                print(line, end="", flush=True)
                log.write(line)
            returncode = process.wait()
        except KeyboardInterrupt:
            import signal
            process.send_signal(signal.SIGINT)
            raise
        finally:
            process.stdout.close()
        print(f"Child exited with code {returncode}; elapsed {time.monotonic() - started:.1f}s; "
              f"console log: {path}", flush=True)
        if returncode:
            raise subprocess.CalledProcessError(returncode, command)
    return returncode

import os
import pathlib
import subprocess
import sys
import urllib.request

assert (3, 11) <= sys.version_info[:2] <= (3, 13), "Use a Python 3.11–3.13 runtime"
release = "v0.2.0"
env_dir = pathlib.Path("/content/captionbert-diagnostic-env")
if not (env_dir / "bin/python").exists():
    # Colab may omit stdlib ensurepip. virtualenv seeds pip without that module.
    bootstrap = pathlib.Path("/content/captionbert-diagnostic-bootstrap")
    run_logged([sys.executable, "-m", "pip", "install", "--target", str(bootstrap),
                    "virtualenv==20.38.0"], log_path="/content/captionbert-logs/setup.log")
    bootstrap_env = os.environ.copy()
    bootstrap_env["PYTHONPATH"] = str(bootstrap)
    run_logged([sys.executable, "-m", "virtualenv", "--no-download", str(env_dir)],
                   env=bootstrap_env, log_path="/content/captionbert-logs/setup.log")
python = str(env_dir / "bin/python")
# Download the three requirements files from the same immutable code tag.
req_dir = pathlib.Path("/content/captionbert-diagnostic-install")
req_dir.mkdir(parents=True, exist_ok=True)
for name in ("requirements.txt", "requirements-cpu.txt"):
    url = f"https://raw.githubusercontent.com/AbstractEyes/captionbert-trainer/{release}/{name}"
    urllib.request.urlretrieve(url, req_dir / name)
run_logged([python, "-m", "pip", "install", "-r", str(req_dir / "requirements-cpu.txt")], log_path="/content/captionbert-logs/setup.log")
run_logged([python, "-m", "pip", "install", "--no-deps",
                f"git+https://github.com/AbstractEyes/captionbert-trainer.git@{release}"], log_path="/content/captionbert-logs/setup.log")
run_logged([python, "-m", "pip", "check"], log_path="/content/captionbert-logs/setup.log")
run_logged([python, "-c", "import sys,torch,captionbert_rotary; print(sys.version); "
                "print(captionbert_rotary.__version__, torch.__version__, torch.version.cuda); "
                "print('CPU diagnostic environment ready')"], log_path="/content/captionbert-logs/setup.log")


The analysis first pins the Hub revision. It skips unfinished arms and reuses
published measurements with matching checkpoint hashes and probe inputs. Run the
analysis cell again later for newly completed arms. It never overwrites original
weights or task results; uploads go under each run's `analyses/` directory.

Measurements: ideal address kernels/coherence/periodic aliases/clamping; identical-content
requested-address ambiguity; frozen address changes and per-band q/k removals;
and per-layer/head score, energy, and attention-mass measurements. The code-only
lookup has an oracle request. It is not a trained semantic-retrieval result.

The cell also prepares the model README in the original v2 card's format. It
publishes only when every arm/seed, all eight task scores and matching frozen
measurements are present. Until then it saves a clearly marked local draft and
prints the missing checks. Original run results remain unchanged.


In [ ]:
def run_logged(command, log_path, env=None):
    import os
    from pathlib import Path
    import subprocess
    import time

    child_env = os.environ.copy() if env is None else dict(env)
    child_env["PYTHONUNBUFFERED"] = "1"
    child_env["PYTHONIOENCODING"] = "utf-8"
    path = Path(log_path)
    path.parent.mkdir(parents=True, exist_ok=True)
    print(f"Streaming child output; console log: {path}", flush=True)
    started = time.monotonic()
    with path.open("a", encoding="utf-8", buffering=1) as log:
        process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                   text=True, encoding="utf-8", errors="replace", bufsize=1,
                                   env=child_env)
        try:
            for line in process.stdout:
                print(line, end="", flush=True)
                log.write(line)
            returncode = process.wait()
        except KeyboardInterrupt:
            import signal
            process.send_signal(signal.SIGINT)
            raise
        finally:
            process.stdout.close()
        print(f"Child exited with code {returncode}; elapsed {time.monotonic() - started:.1f}s; "
              f"console log: {path}", flush=True)
        if returncode:
            raise subprocess.CalledProcessError(returncode, command)
    return returncode

import os
import textwrap
from google.colab import userdata

python = "/content/captionbert-diagnostic-env/bin/python"
child_env = os.environ.copy()
child_env["HF_TOKEN"] = userdata.get("HF_TOKEN")
run_logged([python, "-u", "-c", textwrap.dedent("""
    from captionbert_rotary.analyse import analyse_available
    analyse_available(
        repo='AbstractPhil/captionbert-8192-rotary',
        experiment='817daa9e66a8119dc6b18a9b63fad32490d1c891e5f26750e91125d9e5e14e6c',
        arms=('A0', 'A1', 'A2', 'A3'), seeds=(1, 2),
        device='cpu', threads=2, rows=16, batch_size=4,
        data_dir='/content/captionbert-diagnostic-data',
        output_dir='/content/captionbert-diagnostics', push=True,
    )
    from captionbert_rotary.model_card import finalize_model_card
    finalize_model_card(
        repo='AbstractPhil/captionbert-8192-rotary',
        experiment='817daa9e66a8119dc6b18a9b63fad32490d1c891e5f26750e91125d9e5e14e6c',
        output_dir='/content/captionbert-diagnostics/model-card', push=True,
    )
""")], env=child_env, log_path="/content/captionbert-logs/diagnostics.log")


The compact table is a navigation aid. The JSON reports retain exact probe
texts/tokens/addresses, checkpoint hashes, dtype/device, numerical repeat, all
interventions, and every layer/head. A0 has no rotary bands, so its band-specific
entries are n/a. A1/A3 have no anchor band and should respond by zero to anchor
removal. A2's sensitivity alone does not establish an improvement; compare both
seeds and the original held-out/task metrics separately.


In [ ]:
import json
from pathlib import Path
from IPython.display import Markdown, display

root = Path("/content/captionbert-diagnostics")
summary = json.loads((root / "summary.json").read_text())
lines = ["| Arm/seed | Step | Repeat max error | Sentence-ID change (mean L2) | Anchor removal (mean L2) |",
         "|---|---:|---:|---:|---:|"]
def number(value):
    return f"{value:.6g}" if value is not None else "n/a"
for entry in summary["completed"]:
    report = json.loads(Path(entry["path"]).read_text())
    cases = report["interventions"]
    lines.append(f"| {entry['name']} | {report['identity']['step']} | "
                 f"{number(report['repeat_floor']['max_absolute_change'])} | "
                 f"{number(cases['sentence_ids_collapsed'].get('mean_l2_change'))} | "
                 f"{number(cases['remove_anchor'].get('mean_l2_change'))} |")
display(Markdown("\n".join(lines)))
print("Awaiting completed arms:", summary["awaiting_completed_arms"])
print("Unshipped analyses:", summary.get("shipping", {}).get("pending_runs", []))
print("Full per-layer/head tables and probe inputs:", root)
